**Imports**

Here we import the required libraries. 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader 

import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

**Data Preparation**

At first, we should load the Iris dataset using load_iris() from sklearn.datasets.
Then we declare a variable named "dataset" to set it equal to load_iris()
load_iris() contains different sections, but what WE are interested in, are the two ".data" (X) and ".target" (y) of it.

But they are in fact, NumPy ndarrays. PyTorch needs torch.tensors! So then we convert them.

The Iris dataset contains 150 data instances (rows) and 5 attributes/columns (4 numerical feature columns and 1 target species column).
The dataset breaks down as follows:
Total Samples: 150 records 
Classes (Species): 3 species (Iris setosa, Iris versicolor, and Iris virginica) with exactly 50 samples each
Features: 4 measurements in centimeters (sepal length, sepal width, petal length, and petal width)


In [ ]:
# --- Data Preparation ---
dataset = load_iris()
X, y = dataset.data, dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=4, shuffle=True, stratify=y)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# Class indices for CrossEntropyLoss MUST be integers (torch.long) and 1D (no squeeze/unsqueeze)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

#print () each one to see them, and to get a general sense of the data so that you get familiar with it.

***Now we create the DataLoader***

In [ ]:
# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

**Model Architecture**

After getting a sense of the data, we need to deisgn our neural netowork.
Here, we're going to use a very basic architecture: Multi-layer Perceptron.

Neural networks' architectures are usually defined as classes in PyTorch, which all inherit from nn.Module and also MUST have a "forward" function that defines the flow of data through the layers we define in the __init__() method.

Taking a look at the code below, you can clearly see that each instance of the MLP class should send in_features and out_features, which are basically the number of the features by the help of which we want to classify the flowers. So, in_features here equals to 4 and out_features equals to 3 (the 3 types of iris).

The activation function used here (ReLU) is not the "universal" one used for MLPs. You can try sigmoid and tanh if you want later.

The forward method takes two arguments: self and x. "x" is the data (features, X) and it should flow through the layers we define in the __init__() method.

Note that we can also use nn.LazyLinear instead of nn.Linear if we're not sure about the number of features.

And lastly, we initialiaze a variable "model" as an instance of the class MLP.


In [ ]:
# --- Model Definition ---
class MLP(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear1 = nn.Linear(in_features, 4, bias=True)
        self.linear2 = nn.Linear(4, 3, bias=True)
        # We output to 3 classes (out_features = 3)
        self.outputLayer = nn.Linear(3, out_features)

    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        # NO RELU on the output layer! We want the raw scores (logits).
        return self.outputLayer(x)
    
# Initialize model with 3 output features
model = MLP(in_features=4, out_features=3)

**Hyperparameters**

Before jumping into the training loop, we should set a few hyperparameters.
We can totally put them inside the training loop instead of declaring variables and storing the values there. Makes no difference, but reduces code clarity and reusability.

Here I used the Stochastic Gradient Descent optimizer. You can try torch.optim.Adam too!
Also, I set the loss function to nn.CrossEntropyLoss() becuase this is a classification problem. For regression problems, one should use MSELoss() or etc.

So keep in mind:
Before designing a training loop, we should set the number of epochs, the learning_rate, the optimizer and the loss function. 


In [ ]:
# --- Hyperparameters Setup ---
num_epochs = 100
learning_rate = 0.01 
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# SWITCHED: MSELoss -> CrossEntropyLoss for classification
loss_fn = nn.CrossEntropyLoss()

**Training Loop**

Now it's time for the training loop.

First, I declared two empty lists train_losses and test_losses. Why? To store each epoch's losses in them so that we could plot them later and see what happened. Though we can see the numbers, plots never fail to amaze you!

Then, we put a for loop.
This "first" for loop runs through the whole dataset (NOT each batch of data). Based on the definition, this is called an EPOCH!
Yes, an epoch is one looping through the whole dataset. In other words, it's when the model sees the whole data once.

But you can't fit some huge, gigantic datasets into the whole RAM from your disk, can you?
So, you have to divide it into batches. That's what we did using scikit-learn's amazing tool, train_test_split. 

During each "batch", we should first reset the gradients using optimizer.zero_grad(). Since everytime the loop begins a fresh batch, we want to find the new gradients, we have to remove the previous ones. So, that's why.

Then we'll nedd to feed the "batch" into the model. So, what should we do? Just feed it to the model and set a variable to catch the results.

OK, so far, so good.

Now you need to see how "wrong" the predictions were, right? Afterall, how are you going to "train" your model, huh?
So, go ahead and calculate the loss. How? Using the loss_fn we chose! Just feed it the predictions and the ground truth labels. It does the job for you.

Very good. Now we knoe the "loss" and have managed to calculate it. What should we do with it? 
That's right: you have to backpropagate it to tune the weights and biases that were initially initiliazed randomly.
So, in short: do a .backward() to the loss, and a .step() to the optimizer. 


During each "batch" we're also interested in the specific accuracy and loss of our model during that period. So we store them in a list.
But we should keep in mind that before that, we have to put the model into .eval() mode.
AND also keep in mind that when we're calculating the loss of the model, we're interested in the loss on the "test" section, which the model hasn't seen, not the train_loss (which is important indeed, but not here lol).

The last if ... statement is nothing but a decent way to show the progress of the training process. You can also use libraries like tqdm to make it more nice and tidy :D



In [ ]:
train_losses = []
test_losses = []

# --- Training Loop ---
for epoch in range(num_epochs):
    model.train() #set the model to training mode so that we could adjust and change its weights/biases!
    running_train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        y_pred = model(batch_X) # Output shape: [batch_size, 3]
        loss = loss_fn(y_pred, batch_y) # batch_y shape: [batch_size]
        
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * batch_X.size(0)
        
    epoch_train_loss = running_train_loss / len(X_train)
    train_losses.append(epoch_train_loss) #.append() adds new members to the list we made earlier in this code block.
    
    # Validation
    model.eval()
    with torch.no_grad():
        y_pred_val = model(X_test)
        val_loss = loss_fn(y_pred_val, y_test)
        test_losses.append(val_loss.item())
        
        # Calculate Accuracy for printing
        # torch.max(..., 1) returns (values, indices). The indices are our predicted class integers.
        _, predictions = torch.max(y_pred_val, 1) # _ (underscore) is a valid variable name in Python. It's used when we really don't need some part(s) of the data we get from a function.
        correct = (predictions == y_test).sum().item()
        accuracy = (correct / len(y_test)) * 100

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d} | Train Loss: {epoch_train_loss:.4f} | Test Loss: {val_loss.item():.4f} | Test Acc: {accuracy:.1f}%")


**Plotting**

Here comes plotting.

Everybody hates matplotlib when they begin their journey, but after a while you'll get used to it.

The key is to learn the way it handles stuff and names them, like "figures" instead of canvas.

Anyway.

Here I use a pretty straightforward way to display the plots (using a single figure and not subplots):
plt is "matplotlib.pyplot", we defined it earlier in this notebook as:
import matplotlib.pyplot as plt

so,

plt.figure() creates a figure. We can ask for a specific size (in inches) or just let it do its thing automatically.

plt.plot() ---> Plots lines/curves, and requires a list, array, whatever. Different linestyles exist, you can find out more in their own documentations.

plt.xlabel() and plt.ylabel() ---> Used when we want to set labels for the X and Y axis.

plt.title() ---> Sets a title for the plot (nah really?!)

plt.legend() ---> Enabling this would display the legend. Otherwise it won't be shown.

plt.grid() ---> Accepts different parameters, but the most important one is the alpha here: the less you set it, the whiter/grayer the gridlines get. 

plt.show() ---> Open a new window and display the figure/plot.



In [ ]:

# --- Plotting ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss', color='royalblue', linewidth=2)
plt.plot(range(1, num_epochs + 1), test_losses, label='Test Loss', color='darkorange', linewidth=2, linestyle='--')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.title('Multi-Class Classification Loss over Epochs')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()



**BONUS**

Visualizing the model architecture using visualtorch



In [ ]:

import visualtorch

# Assuming 'model' is our instantiated MLP(in_features=4, out_features=3)
input_shape = (1, 4)

# ----------------------------------------------------
# Style: Graph View (Best for MLPs - draws actual neurons)
# ----------------------------------------------------
# Pass style="graph" and your specific node customization options directly.

graph_img = visualtorch.render(
    model, 
    input_shape=input_shape,
    style="graph",
    node_size=30,
    layer_spacing=150,
    node_spacing=15
)
dpi = 9000
plt.figure(figsize=(graph_img.width/dpi, graph_img.height/dpi), dpi=dpi)
plt.imshow(graph_img)
plt.axis("off")
plt.title("Architecture (Graph Style)")
plt.tight_layout() #optional
plt.show()

